In [1]:
!pip install transformers gradio accelerate sentencepiece torch --quiet


In [2]:
from transformers import pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name,
    torch_dtype="auto",
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0


In [3]:
def summarize_with_prompt(text, max_new_tokens=200):
    prompt = (
        "You are an expert abstractive summarization assistant. "
        "Rewrite the following text as a short, concise, high-quality abstractive summary.\n\n"
        f"Text:\n{text}\n\nSummary:"
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
        eos_token_id=generator.tokenizer.eos_token_id
    )

    # Remove the prompt from the output
    result = output[0]["generated_text"][len(prompt):].strip()
    return result


In [4]:
import gradio as gr

def summarize_interface(input_text, max_tokens):
    if not input_text:
        return "Please enter text."
    return summarize_with_prompt(input_text, max_new_tokens=int(max_tokens))

demo = gr.Interface(
    fn=summarize_interface,
    inputs=[
        gr.Textbox(lines=12, label="Input Text"),
        gr.Slider(50, 400, value=200, label="Max Summary Tokens")
    ],
    outputs=gr.Textbox(lines=10, label="Abstractive Summary"),
    title="Decoder-Only Abstractive Summarizer (Pipeline + Prompt)",
    description="Summarization using a decoder-only transformer with custom prompt + HF pipeline."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a3bf2e37c53c947e68.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
